[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amistein/ml-study-group/blob/main/notebooks/lesson_008.ipynb)

### Matrices and Batched Operations

In the last lesson, we took the weighted sum operation from our `predict` function and gave it a proper name: the dot product. We also introduced NumPy, which can compute dot products (and other vector operations) much faster than a Python loop. But we were still predicting one house at a time. We're taking a single house's features, dot them with the weights, add the base price, and get one prediction. If we had 1000 houses, we'd need to loop and call np.dot 1000 times. There's a way to predict all 1000 houses in a single operation, and understanding how it works introduces one of the most important structures in machine learning: the **matrix**.

A **matrix** is a grid of numbers arranged in rows and columns. We've already been working with something that looks like one... our housing data.

In [ ]:
houses = [
    [1500, 3, 2, 10],
    [2000, 4, 3, 20],
    [800,  2, 1, 40],
    [2500, 5, 3, 5]
]

That's a matrix with 4 rows and 4 columns. Each row is one house (a vector of features), and each column is one feature across all houses (all the square footages, all the bedroom counts, etc.). We describe its size as "4 by 4" or "4x4". A matrix doesn't have to be square though. If we had 100 houses with 4 features each, that would be a 100x4 matrix (100 rows, 4 columns).

So what does it mean to "multiply" a matrix (the housing data) by a vector (the weights)? It means: take the dot product of each row of the matrix with the vector, and collect all the results into a new vector. So if we multiply our 4x4 housing matrix by our weight vector (length 4), we get a vector of 4 predictions, one per house. Each prediction is the dot product of that house's features with the weights, which is exactly what our predict function computes (minus the base price). We just do all 4 at once instead of one at a time.

Let's trace through it. Our weight vector is [100, 10000, 5000, -3000]. The first row of our matrix is [1500, 3, 2, 10]. The dot product is 1500\*100 + 3\*10000 + 2\*5000 + 10\*(-3000) = 160000. The second row is [2000, 4, 3, 20], giving 2000\*100 + 4\*10000 + 3\*5000 + 20\*(-3000) = 195000. And so on for each row. The result is the vector [160000, 195000, -15000, 300000].

In NumPy, this operation is written with the `@` operator:

In [ ]:
predictions = houses @ weights

That single line replaces a loop over all houses. And because NumPy runs the entire matrix-vector multiply in optimized C code, it's dramatically faster than a Python loop, especially when you have thousands or millions of rows. This is why matrices matter in ML. Every training step involves making predictions for a batch of examples, computing a loss, and updating the weights. The more efficiently you can make predictions across a batch, the faster training goes.

One detail about dimensions. When you multiply a matrix by a vector, the number of columns in the matrix must equal the length of the vector. Our 4x4 matrix has 4 columns, and our weight vector has 4 elements, so it works. If the matrix were 4x3 (4 houses, 3 features), the weight vector would need to have 3 elements. The result always has one element per row of the matrix. So a 4x4 matrix times a length-4 vector gives a length-4 result. A 100x4 matrix times a length-4 vector gives a length-100 result.

In the exercises, we'll do something significant. Right now, predicting prices for all our houses means looping and calling np.dot once per house, then looping again to compute errors. By the end of these exercises, the entire pipeline will be a single expression: features_matrix @ weights + base_price for all predictions, then element-wise subtraction, squaring, and np.mean for the loss. No Python loops at all. This is how real ML code works.

### Exercises
#### Implement the following functions, and ensure the tests pass.

*Reading through the test cases is a good way to make sure you understand what your code is doing.*

**Exercise 1**: Implement matrix-vector multiplication using plain Python loops (no NumPy). This is the same as computing the dot product of each row with the vector, collected into a list. This is purely to make sure you understand what the operation does before we hand it off to NumPy.

In [ ]:
def matrix_vector_multiply(matrix, vector):
    """Multiply a matrix by a vector using plain Python loops.

    Args:
        matrix: list of lists (each inner list is a row of numbers)
        vector: list of numbers (same length as each row)
    Returns:
        list of numbers (one per row of the matrix)
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# --- Tests (do not modify) ---
_mv1 = matrix_vector_multiply(
    [[1, 2, 3],
     [4, 5, 6]],
    [10, 20, 30]
)
assert _mv1 == [140, 320], f"Expected [140, 320], got {_mv1}"

_mv2 = matrix_vector_multiply(
    [[1500, 3, 2, 10],
     [2000, 4, 3, 20],
     [800,  2, 1, 40],
     [2500, 5, 3, 5]],
    [100, 10000, 5000, -3000]
)
assert _mv2 == [160000, 195000, -15000, 300000], f"Expected [160000, 195000, -15000, 300000], got {_mv2}"

# Single-row matrix
_mv3 = matrix_vector_multiply([[3, 4]], [5, 6])
assert _mv3 == [39], f"Expected [39], got {_mv3}"

# Zeros vector
_mv4 = matrix_vector_multiply([[1, 2], [3, 4]], [0, 0])
assert _mv4 == [0, 0], f"Expected [0, 0], got {_mv4}"

print("✅ All tests passed!")

**Exercise 2**: Now do the same thing using NumPy's @ operator. Verify that it produces the same results as your loop version.

In [ ]:
import numpy as np

def matrix_vector_multiply_np(matrix, vector):
    """Multiply a matrix by a vector using NumPy.

    Args:
        matrix: list of lists or 2D NumPy array
        vector: list or 1D NumPy array
    Returns:
        1D NumPy array (one element per row of the matrix)
    """
    # YOUR CODE HERE
    # Hint: convert to NumPy arrays with np.array(), then use the @ operator.
    raise NotImplementedError()


# --- Tests (do not modify) ---
_mnp1 = matrix_vector_multiply_np(
    [[1, 2, 3],
     [4, 5, 6]],
    [10, 20, 30]
)
assert list(_mnp1) == [140, 320], f"Expected [140, 320], got {list(_mnp1)}"

_mnp2 = matrix_vector_multiply_np(
    [[1500, 3, 2, 10],
     [2000, 4, 3, 20],
     [800,  2, 1, 40],
     [2500, 5, 3, 5]],
    [100, 10000, 5000, -3000]
)
assert list(_mnp2) == [160000, 195000, -15000, 300000], f"Got {list(_mnp2)}"

# Verify it matches the loop version
_test_matrix = [[3, 7, 11], [2, 5, 8], [1, 4, 9]]
_test_vector = [6, 3, 2]
_loop_result = matrix_vector_multiply(_test_matrix, _test_vector)
_np_result = list(matrix_vector_multiply_np(_test_matrix, _test_vector))
assert _loop_result == _np_result, f"Loop gave {_loop_result}, NumPy gave {_np_result}"

print("✅ All tests passed!")

**Exercise 3**: Now let's use this to make predictions for all houses at once. Rewrite the predict function so it takes a matrix of features (one row per house) and returns a vector of predictions, using a single @ operation instead of a loop.

In [ ]:
def predict_batch(features_matrix, weights, base_price):
    """Predict sale prices for multiple houses at once.

    Args:
        features_matrix: 2D NumPy array, shape (num_houses, num_features)
        weights: 1D NumPy array, shape (num_features,)
        base_price: a single number
    Returns:
        1D NumPy array of predicted prices, shape (num_houses,)
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# --- Tests (do not modify) ---
_houses = np.array([
    [1500, 3, 2, 10],
    [2000, 4, 3, 20],
    [800,  2, 1, 40],
    [2500, 5, 3, 5],
])
_weights = np.array([100, 10000, 5000, -3000])
_base_price = 50000

_batch_preds = predict_batch(_houses, _weights, _base_price)

# These should match what you'd get predicting one house at a time
assert list(_batch_preds) == [210000, 245000, 35000, 350000], f"Got {list(_batch_preds)}"

# Verify against single predictions
def _predict_single(features, weights, base_price):
    return np.dot(features, weights) + base_price

for i in range(len(_houses)):
    _single = _predict_single(_houses[i], _weights, _base_price)
    assert _batch_preds[i] == _single, f"House {i}: batch={_batch_preds[i]}, single={_single}"

print(f"Predictions: {list(_batch_preds)}")
print("✅ All tests passed!")

**Exercise 4**: Now compute the MSE loss across the full batch in one shot. No loops. Use the batch predictions, subtract the actual prices (element-wise), square, and take the mean. You did something very similar in lesson 7's mse_numpy, but now the predictions come from predict_batch.

In [ ]:
def batch_mse(features_matrix, weights, base_price, actual_prices):
    """Compute MSE loss for a batch of predictions.

    Args:
        features_matrix: 2D NumPy array, shape (num_houses, num_features)
        weights: 1D NumPy array, shape (num_features,)
        base_price: a single number
        actual_prices: 1D NumPy array of actual sale prices
    Returns:
        mean squared error (a single number)
    """
    # YOUR CODE HERE
    # Hint: use predict_batch, then compute MSE with element-wise operations.
    raise NotImplementedError()


# --- Tests (do not modify) ---
_actual_prices = np.array([300000, 400000, 150000, 500000])

_loss = batch_mse(_houses, _weights, _base_price, _actual_prices)
assert _loss == 16962500000.0, f"Expected 16962500000.0, got {_loss}"

# Perfect predictions should give 0 loss
_perfect_prices = predict_batch(_houses, _weights, _base_price)
_loss_perfect = batch_mse(_houses, _weights, _base_price, _perfect_prices)
assert _loss_perfect == 0.0, f"Expected 0.0, got {_loss_perfect}"

print(f"MSE loss: {_loss:,.0f}")
print("✅ All tests passed!")